In [1]:
import pandas as pd
import numpy as np
from scipy.signal import argrelextrema
from scipy.signal import find_peaks
from itertools import product
from tqdm import tqdm
import time
import numba as nb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import talib as ta
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df=pd.read_csv('../../all_data_EUR_USD.csv')

In [3]:
df.set_index('time',inplace=True)

In [4]:
df_test=df.copy()

In [5]:
def calc_low_points(low_vals):

    temp_min=low_vals[0]
    

    for i in range(len(low_vals)):

         if low_vals[i]<temp_min:

             temp_min=low_vals[i]


        

    if temp_min==low_vals[-2]:

       return low_vals[-2]

    else:

        return -101

In [6]:
def calc_high_points(high_vals):

    temp_max=high_vals[0]
    

    for i in range(len(high_vals)):

         if high_vals[i]>temp_max:

             temp_max=high_vals[i]


        

    if temp_max==high_vals[-2]:

       return high_vals[-2]

    else:

        return 101

In [7]:
def find_ll(stoch):


    temp_min=-101
   

    for i in range(len(stoch) - 2, -1, -1):
        if stoch[i]!=-101:
       
            if stoch[i]<stoch[-1]:
    
                temp_min=stoch[i]
                
                break

               
    
    return temp_min

In [8]:
def find_hh_idx(stoch):

    
    last_max_idx=0

    for i in range(len(stoch) - 2, -1, -1):
        if stoch[i]!=101:
        
            if stoch[i]>stoch[-1]:
        
                
                last_max_idx=i-1
                break

               
    
    return last_max_idx

In [9]:
def find_ll_idx(stoch):

    
    last_min_idx=0

    for i in range(len(stoch) - 2, -1, -1):
        if stoch[i]!=-101:
        
            if stoch[i]<stoch[-1]:
    
                
                last_min_idx=i-1
                break

               
    
    return last_min_idx

In [10]:
def find_hh(stoch):

    temp_max=101

    for i in range(len(stoch) - 2, -1, -1):
        if stoch[i]!=101:
       
            if stoch[i]>stoch[-1]:
    
                temp_max=stoch[i]
                
                break

               
    
    return temp_max

In [11]:
class STOCH_divergence():

    def __init__(self,data):

        self.data=data

        self.params_range={'fastk_period':[2,5,9],
                           'slowk_period':[1,3,7],
                           'slowd_period':[1,3,7],
                          'lookback':[57,91],
                          'line':['k','d']}

        self.possible_strats={'strategy_desc':'Strategy STOCH divergence',
                              'div_point':{'name':'detect divergence STOCH points',
                                          'positions':{'buy':'bull point',
                                                       'sell':'bear point'},
                                            'pos_columns':{}}}



    def create_params_combs(self):

        return list(product(*self.params_range.values()))

    def calc_indicator(self, fastk,slowk,slowd, lookback, line='k'):


        self.fastk=fastk
        self.slowk=slowk
        self.slowd=slowd
        self.lookback=lookback
        self.line=line
        
        df=self.data.copy()

        
        df['STOCH']= ta.STOCH(df['h'], df['l'], df['c'], fastk_period=fastk, slowk_period=slowk, slowk_matype=0, slowd_period=slowd, slowd_matype=0)[0]
            
        if line=='d':

            df['STOCH']= ta.STOCH(df['h'], df['l'], df['c'], fastk_period=fastk, slowk_period=slowk, slowk_matype=0, slowd_period=slowd, slowd_matype=0)[1]

            
        
        
        
        
        
             
        
        
        
        df['ll']=df['STOCH'].rolling(3).apply(calc_low_points, raw=True, engine='numba')
        df['hh']=df['STOCH'].rolling(3).apply(calc_high_points, raw=True, engine='numba')
        
        
        
        df['idx']=range(len(df))
        
        df['last_min']=df['ll'].rolling(lookback).apply(find_ll, raw=True, engine='numba')
        df['last_max']=df['hh'].rolling(lookback).apply(find_hh, raw=True, engine='numba')
        
        df['prev_min_idx']=df['ll'].rolling(lookback).apply(find_ll_idx, raw=True, engine='numba')+df.idx-lookback+1
        df['prev_max_idx']=df['hh'].rolling(lookback).apply(find_hh_idx, raw=True, engine='numba')+df.idx-lookback+1
        df['last_min_idx']=np.where((df['ll']!=-101)&(df['last_min']!=-101)&(df['ll'].notna()), df['idx'].shift(),0)
        df['last_max_idx']=np.where((df['hh']!=101)&(df['last_max']!=101)&(df['hh'].notna()), df['idx'].shift(),0)
        df['last_min_price']=np.where((df['ll']!=-101)&(df['last_min']!=-101)&(df['ll'].notna()), df['l'].shift(),0)
        df['last_max_price']=np.where((df['hh']!=101)&(df['last_max']!=101)&(df['hh'].notna()), df['h'].shift(),0)
        
        prev_min_prices_idx=np.array(df['prev_min_idx'])
        prev_max_prices_idx=np.array(df['prev_max_idx'])
        prev_min_prices=np.array(df['l'].iloc[np.nan_to_num(prev_min_prices_idx)])
        prev_max_prices=np.array(df['h'].iloc[np.nan_to_num(prev_max_prices_idx)])
        df['prev_min_prices']=prev_min_prices
        df['prev_max_prices']=prev_max_prices
        
        self.data=df.copy()


    def calc_position(self):

        df=self.data.copy()

        self.pos_ch_colname=f'pos_ch_div_point_fk_{self.fastk}_sk_{self.slowk}_sd_{self.slowd}_lb_{self.lookback}_{self.line}'
        self.pos_colname=f'pos_div_point_fk_{self.fastk}_sk_{self.slowk}_sd_{self.slowd}_lb_{self.lookback}_{self.line}'

        cond_buy_check=(df['ll']!=-101)&(df['last_min']!=-101)&(df['prev_min_idx'].notna())
        cond_buy_price=(df['last_min_price']!=0)&(df['prev_min_prices']>df['last_min_price'])
        cond_sell_check=(df['hh']!=101)&(df['last_max']!=101)&(df['prev_max_idx'].notna())
        cond_sell_price=(df['last_max_price']!=0)&(df['prev_max_prices']<df['last_max_price'])

        df[f'{self.pos_ch_colname}']=np.nan
        df[f'{self.pos_ch_colname}']=np.where(cond_buy_check&cond_buy_price,1,df[f'{self.pos_ch_colname}'])
        df[f'{self.pos_ch_colname}']=np.where(cond_sell_check&cond_sell_price,-1,df[f'{self.pos_ch_colname}'])
        df[f'{self.pos_colname}']= df[f'{self.pos_ch_colname}'].ffill()
        df[f'{self.pos_ch_colname}']=df[f'{self.pos_ch_colname}'].fillna(0)
        df[f'{self.pos_colname}']= df[f'{self.pos_colname}'].fillna(0)
        
        

        self.data=df[[col for col in df.columns if col not in ['ll','hh','idx','last_min','STOCH',
                                                              'last_max','prev_min_idx','prev_max_idx','last_min_idx','last_max_idx','last_min_price','last_max_price',
                                                              'prev_min_prices','prev_max_prices']]]



    

In [12]:
sd=STOCH_divergence(df_test)

In [13]:
cmb=sd.create_params_combs()

In [14]:
#sd.calc_indicator(5,3,3,30,'d')

In [15]:
#sd.data[50:100].loc[:,['idx','h','STOCH','hh','last_max','last_max_idx','prev_max_idx','last_max_price','prev_max_prices']]

In [16]:
#sd.calc_position()

In [17]:
#sd.data[50:100]

In [18]:
#sd.data['pos_ch_div_point_fk_5_sk_3_sd_3_lb_30_d'].value_counts()

In [19]:
for c in tqdm(cmb):

    

    sd.calc_indicator(*c)
    
    sd.calc_position()

100%|██████████| 108/108 [05:31<00:00,  3.07s/it]


In [20]:
sd.data.to_csv('STOCH_divergence_data.csv')

In [21]:
with open('sd_divergence.json', "w") as f:
    json.dump(sd.possible_strats, f)